# Bee Around precompute (sample run)

This notebook builds a **small test precompute** for the first two comparison ideas:
1. Signature Species (local share vs global share)
2. Place Rank / Percentiles Against Peers

It is intentionally small first (few countries + few cities) so we can validate quality and runtime before scaling.

In [ ]:
import json
import math
import random
import time
from pathlib import Path
from typing import Dict, List, Optional

import requests

GBIF_BASE = "https://api.gbif.org/v1"
NOMINATIM_BASE = "https://nominatim.openstreetmap.org"

GBIF_TIMEOUT_SEC = 60
GBIF_MAX_RETRIES = 4
GBIF_RETRY_BACKOFF_SEC = 1.0
GBIF_REQUEST_SLEEP_SEC = 0.15  # pacing between successful GBIF calls

# Nominatim usage policy: <= 1 request per second, custom UA.
NOMINATIM_REQUEST_SLEEP_SEC = 1.1
NOMINATIM_TIMEOUT_SEC = 30
NOMINATIM_MAX_RETRIES = 3

# Persist all outputs in kernel home so reruns resume safely.
OUTPUT_DIR = Path.home() / "bee_around_precompute"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
NOMINATIM_CACHE_PATH = OUTPUT_DIR / "nominatim_cache.json"
OUTPUT_JSON_PATH = OUTPUT_DIR / "comparison_precompute.json"
GLOBAL_BASELINE_PATH = OUTPUT_DIR / "global_baseline.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "bee-around-precompute/0.1 (contact: bee-around)"})

print("Output dir:", OUTPUT_DIR)

In [ ]:
def parse_retry_after_seconds(value: Optional[str]) -> Optional[float]:
    if not value:
        return None

    try:
        return max(0.0, float(value))
    except ValueError:
        return None


def gbif_occurrence_search(params: Dict) -> Dict:
    url = f"{GBIF_BASE}/occurrence/search"
    last_error: Optional[Exception] = None

    for attempt in range(GBIF_MAX_RETRIES + 1):
        try:
            response = session.get(url, params=params, timeout=GBIF_TIMEOUT_SEC)

            if response.status_code == 429:
                retry_after = parse_retry_after_seconds(response.headers.get("Retry-After"))
                wait_sec = retry_after if retry_after is not None else GBIF_RETRY_BACKOFF_SEC * (attempt + 1)
                wait_sec += random.uniform(0, 0.25)  # small jitter to avoid synchronized retries
                if attempt >= GBIF_MAX_RETRIES:
                    response.raise_for_status()
                time.sleep(wait_sec)
                continue

            if 500 <= response.status_code < 600:
                if attempt >= GBIF_MAX_RETRIES:
                    response.raise_for_status()
                wait_sec = GBIF_RETRY_BACKOFF_SEC * (attempt + 1)
                wait_sec += random.uniform(0, 0.25)
                time.sleep(wait_sec)
                continue

            response.raise_for_status()
            time.sleep(GBIF_REQUEST_SLEEP_SEC)
            return response.json()

        except requests.RequestException as exc:
            last_error = exc
            if attempt >= GBIF_MAX_RETRIES:
                raise
            wait_sec = GBIF_RETRY_BACKOFF_SEC * (attempt + 1)
            wait_sec += random.uniform(0, 0.25)
            time.sleep(wait_sec)

    if last_error:
        raise last_error
    raise RuntimeError("GBIF request failed unexpectedly without exception.")


def occurrence_facets(
    *,
    facet_fields: List[str],
    facet_limit: int = 100,
    country: Optional[str] = None,
    bbox: Optional[Dict[str, float]] = None,
    iucn_categories: Optional[List[str]] = None,
    species_keys: Optional[List[int]] = None,
) -> Dict:
    params: Dict = {
        "limit": 0,
        "facet": facet_fields,
        "facetLimit": facet_limit,
    }

    if country:
        params["country"] = country

    if bbox:
        params["decimalLatitude"] = f"{bbox['minLat']},{bbox['maxLat']}"
        params["decimalLongitude"] = f"{bbox['minLon']},{bbox['maxLon']}"

    if iucn_categories:
        params["iucnRedListCategory"] = iucn_categories

    if species_keys:
        params["speciesKey"] = species_keys

    return gbif_occurrence_search(params)


def normalize_field(value: str) -> str:
    return value.replace("_", "").lower()


def get_facet_counts(payload: Dict, field: str) -> List[Dict]:
    target = normalize_field(field)
    for facet in payload.get("facets", []):
        if normalize_field(str(facet.get("field", ""))) == target:
            return facet.get("counts", [])
    return []

In [ ]:
# Nominatim resolution mirrors the app's CitySearch flow so bbox matches
# exactly what users get when they pick a place in Bee Around.
#
# App reference: frontend src/api/nominatim.ts (searchCities).

def _load_nominatim_cache() -> Dict[str, dict]:
    if NOMINATIM_CACHE_PATH.exists():
        try:
            return json.loads(NOMINATIM_CACHE_PATH.read_text(encoding='utf-8'))
        except json.JSONDecodeError:
            return {}
    return {}

def _save_nominatim_cache(cache: Dict[str, dict]) -> None:
    NOMINATIM_CACHE_PATH.write_text(json.dumps(cache, indent=2), encoding='utf-8')

_nominatim_cache: Dict[str, dict] = _load_nominatim_cache()
_last_nominatim_call_at: float = 0.0

def _nominatim_throttle() -> None:
    global _last_nominatim_call_at
    elapsed = time.monotonic() - _last_nominatim_call_at
    if elapsed < NOMINATIM_REQUEST_SLEEP_SEC:
        time.sleep(NOMINATIM_REQUEST_SLEEP_SEC - elapsed)
    _last_nominatim_call_at = time.monotonic()

def nominatim_search(query: str, *, language: str = 'en', limit: int = 6) -> List[dict]:
    last_error: Optional[Exception] = None
    for attempt in range(NOMINATIM_MAX_RETRIES + 1):
        _nominatim_throttle()
        try:
            response = session.get(
                f'{NOMINATIM_BASE}/search',
                params={
                    'format': 'json',
                    'q': query,
                    'limit': limit,
                    'addressdetails': 1,
                    'accept-language': language,
                },
                timeout=NOMINATIM_TIMEOUT_SEC,
            )
            if response.status_code == 429:
                if attempt >= NOMINATIM_MAX_RETRIES:
                    response.raise_for_status()
                time.sleep(NOMINATIM_REQUEST_SLEEP_SEC * (attempt + 1))
                continue
            response.raise_for_status()
            return response.json()
        except requests.RequestException as exc:
            last_error = exc
            if attempt >= NOMINATIM_MAX_RETRIES:
                raise
            time.sleep(NOMINATIM_REQUEST_SLEEP_SEC * (attempt + 1))
    if last_error:
        raise last_error
    raise RuntimeError('Nominatim request failed unexpectedly without exception.')

def choose_best_nominatim_result(results: List[dict]) -> Optional[dict]:
    if not results:
        return None
    return sorted(
        results,
        key=lambda r: (
            0 if r.get('class') == 'boundary' else 1,
            -(float(r.get('importance') or 0)),
        ),
    )[0]

def parse_bbox(result: dict) -> Optional[Dict[str, float]]:
    bbox = result.get('boundingbox')
    if not bbox or len(bbox) != 4:
        return None
    south, north, west, east = [float(v) for v in bbox]
    if south >= north or west >= east:
        return None
    return {'minLat': south, 'maxLat': north, 'minLon': west, 'maxLon': east}

def resolve_place(entry: Dict, *, language: str = 'en') -> dict:
    query = entry['query']
    cache_key = f"{language}:{query}"
    cached = _nominatim_cache.get(cache_key)
    if cached:
        return cached

    results = nominatim_search(query, language=language)
    best = choose_best_nominatim_result(results)
    if not best:
        raise ValueError(f'No Nominatim result for {query!r}')
    bbox = parse_bbox(best)
    if not bbox:
        raise ValueError(f'Nominatim result for {query!r} has no usable bbox')

    place = {
        'id': entry.get('id') or query.lower().replace(' ', '-'),
        'label': best.get('display_name') or query,
        'query': query,
        'kind': entry.get('kind', 'city'),
        'latitude': float(best['lat']),
        'longitude': float(best['lon']),
        'bbox': bbox,
        'countryCode': (best.get('address') or {}).get('country_code', '').upper() or None,
        'nominatim': {
            'osmType': best.get('osm_type'),
            'osmId': best.get('osm_id'),
            'class': best.get('class'),
            'type': best.get('type'),
            'importance': best.get('importance'),
        },
    }
    _nominatim_cache[cache_key] = place
    _save_nominatim_cache(_nominatim_cache)
    return place

def bbox_area_km2(bbox: Dict[str, float]) -> float:
    lat_mid = (bbox['minLat'] + bbox['maxLat']) / 2
    lat_km = 111.32
    lon_km = 111.32 * math.cos(math.radians(lat_mid))
    width = max(0.0, (bbox['maxLon'] - bbox['minLon']) * lon_km)
    height = max(0.0, (bbox['maxLat'] - bbox['minLat']) * lat_km)
    return width * height

In [ ]:
# Load full PLACE_ENTRIES from CSV inputs when available.
# Falls back to the inline 6-place sample list if the CSVs are missing,
# so the notebook stays runnable end-to-end without the CSV files.
#
# Stable `id` (iso2 / slug) is what we use in checkpoint filenames so a
# Nominatim re-resolve does not invalidate prior work.
import csv

# CSVs live directly under kernel home precompute folder: ~/bee_around_precompute/*.csv
COUNTRIES_CSV_PATH = OUTPUT_DIR / 'countries.csv'
CITIES_CSV_PATH = OUTPUT_DIR / 'cities.csv'

def _load_countries_csv(path: Path) -> List[Dict]:
    entries: List[Dict] = []
    with path.open(encoding='utf-8') as f:
        for row in csv.DictReader(f):
            query = (row.get('query') or row.get('name') or '').strip()
            iso2 = (row.get('iso2') or '').strip().lower()
            if not query or not iso2:
                continue
            entries.append({'query': query, 'kind': 'country', 'id': iso2})
    return entries

def _load_cities_csv(path: Path) -> List[Dict]:
    entries: List[Dict] = []
    with path.open(encoding='utf-8') as f:
        for row in csv.DictReader(f):
            query = (row.get('query') or '').strip()
            slug = (row.get('slug') or '').strip().lower()
            if not query or not slug:
                continue
            entries.append({
                'query': query,
                'kind': 'city',
                'id': slug,
                'note': (row.get('note') or '').strip() or None,
            })
    return entries

if COUNTRIES_CSV_PATH.exists() and CITIES_CSV_PATH.exists():
    countries = _load_countries_csv(COUNTRIES_CSV_PATH)
    cities = _load_cities_csv(CITIES_CSV_PATH)
    PLACE_ENTRIES = countries + cities
    print(f'Loaded PLACE_ENTRIES from CSVs: {len(countries)} countries + {len(cities)} cities = {len(PLACE_ENTRIES)} total')
else:
    print('CSV inputs not found; keeping inline sample PLACE_ENTRIES')
    print('  expected:', COUNTRIES_CSV_PATH)
    print('  expected:', CITIES_CSV_PATH)

print('first 3:', PLACE_ENTRIES[:3])
print('last 3: ', PLACE_ENTRIES[-3:])

In [ ]:
THREAT_CATS = ['VU', 'EN', 'CR']

EARTH_RADIUS_KM = 6371.0088

def bbox_area_km2(bbox: Dict[str, float]) -> float:
    """Approximate the area of a lat/lon bbox in km^2. Good enough for ranking."""
    lat_min = math.radians(bbox['minLat'])
    lat_max = math.radians(bbox['maxLat'])
    lon_delta = math.radians(bbox['maxLon'] - bbox['minLon'])
    area_sr = abs(math.sin(lat_max) - math.sin(lat_min)) * abs(lon_delta)
    return area_sr * (EARTH_RADIUS_KM ** 2)


def count_unique_species(bbox: Dict[str, float], *, page_size: int = 1000, hard_cap: int = 200000) -> int:
    """Real unique-species count for a bbox by paginating the speciesKey
    facet. Stops when a page returns fewer than `page_size` rows (last
    page) or when the running total reaches `hard_cap` (defensive).
    Replaces `len(species_counts)` which was silently capped at
    `facetLimit=1000` and saturated the percentile ranking."""
    total = 0
    offset = 0
    while True:
        params = {
            "limit": 0,
            "facet": "speciesKey",
            "facetLimit": page_size,
            "facetOffset": offset,
            "decimalLatitude": f"{bbox['minLat']},{bbox['maxLat']}",
            "decimalLongitude": f"{bbox['minLon']},{bbox['maxLon']}",
        }
        payload = gbif_occurrence_search(params)
        page = get_facet_counts(payload, "speciesKey")
        total += len(page)
        if len(page) < page_size or total >= hard_cap:
            break
        offset += page_size
    return total


def compute_place_metrics(
    *,
    bbox: Dict[str, float],
    species_facet_limit: int = 1000,
) -> Dict:
    """Same call shape the app makes: bbox-only, facetted occurrence search."""
    facets = occurrence_facets(
        facet_fields=['speciesKey', 'iucnRedListCategory'],
        facet_limit=species_facet_limit,
        bbox=bbox,
    )

    total_records = facets.get('count', 0)
    species_counts = get_facet_counts(facets, 'speciesKey')
    iucn_counts = get_facet_counts(facets, 'iucnRedListCategory')

    # Paginated count — len(species_counts) is capped at species_facet_limit
    # and saturates the percentile for most places. Use a real count.
    unique_species = count_unique_species(bbox)
    iucn_map = {c.get('name'): c.get('count', 0) for c in iucn_counts}
    threatened = sum(iucn_map.get(cat, 0) for cat in THREAT_CATS)
    assessed = sum(iucn_map.values())

    area_km2 = bbox_area_km2(bbox)
    threatened_share = (threatened / assessed) if assessed > 0 else 0.0
    records_per_km2 = (total_records / area_km2) if area_km2 > 0 else None

    return {
        'totalRecords': total_records,
        'uniqueSpecies': unique_species,
        'areaKm2': area_km2,
        'recordsPerKm2': records_per_km2,
        'threatenedShare': threatened_share,
        'iucnCounts': iucn_map,
        'topSpecies': [
            {'speciesKey': int(s['name']), 'count': s['count']}
            for s in species_counts
            if str(s.get('name', '')).isdigit()
        ],
    }


In [ ]:
# Global baseline for Idea 1 (Signature Species).
# Persisted to disk so reruns don't re-pull the heavy global facet.

def build_global_baseline(*, species_facet_limit: int = 500) -> Dict:
    facets = occurrence_facets(
        facet_fields=['speciesKey', 'month'],
        facet_limit=species_facet_limit,
    )
    total = facets.get('count', 0)
    species_counts = get_facet_counts(facets, 'speciesKey')
    month_counts = get_facet_counts(facets, 'month')

    species_map = {
        int(item['name']): item['count']
        for item in species_counts
        if str(item.get('name', '')).isdigit()
    }
    month_map = {
        int(m['name']): m['count']
        for m in month_counts
        if str(m.get('name', '')).isdigit()
    }
    month_array = [month_map.get(i, 0) for i in range(1, 13)]

    return {
        'totalRecords': total,
        'month': month_array,
        'topSpeciesGlobal': [
            {'speciesKey': k, 'count': c}
            for k, c in sorted(species_map.items(), key=lambda x: x[1], reverse=True)
        ],
    }

def load_or_build_global_baseline(*, force: bool = False, species_facet_limit: int = 500) -> Dict:
    if not force and GLOBAL_BASELINE_PATH.exists():
        return json.loads(GLOBAL_BASELINE_PATH.read_text(encoding='utf-8'))
    baseline = build_global_baseline(species_facet_limit=species_facet_limit)
    GLOBAL_BASELINE_PATH.write_text(json.dumps(baseline), encoding='utf-8')
    return baseline

global_baseline = load_or_build_global_baseline()
global_total = global_baseline['totalRecords']
global_species_map = {int(row['speciesKey']): int(row['count']) for row in global_baseline['topSpeciesGlobal']}

print('Global total records:', global_total)
print('Global top-species rows:', len(global_species_map))
print('Global month sum:', sum(global_baseline['month']))

In [ ]:
def percentile_rank(values: List[float], x: float) -> float:
    if not values:
        return 0.0
    less_or_equal = sum(1 for v in values if v <= x)
    return less_or_equal / len(values)


def signature_species(local_top_species: List[Dict], global_species_map: Dict[int, int], local_total: int, global_total: int, min_local_count: int = 20, top_n: int = 3) -> List[Dict]:
    out = []
    if local_total <= 0 or global_total <= 0:
        return out

    for row in local_top_species:
        sk = int(row["speciesKey"])
        local_count = row["count"]
        if local_count < min_local_count:
            continue

        global_count = global_species_map.get(sk, 0)
        if global_count <= 0:
            continue

        local_share = local_count / local_total
        global_share = global_count / global_total
        ratio = local_share / global_share if global_share > 0 else None

        if ratio is None:
            continue

        out.append({
            "speciesKey": sk,
            "localCount": local_count,
            "globalCount": global_count,
            "localShare": local_share,
            "globalShare": global_share,
            "overRepresentationRatio": ratio,
        })

    out.sort(key=lambda x: x["overRepresentationRatio"], reverse=True)
    return out[:top_n]

In [ ]:
# Checkpointed per-place run. Safe to interrupt; rerun resumes from where it left off.
#
# - One JSON file per place under CHECKPOINT_DIR.
# - Filename uses entry['id'] (iso2 for countries, slug for cities) so a
#   Nominatim re-resolve doesn't invalidate the checkpoint.
# - Skips any place whose checkpoint already exists.
# - Drops places without a usable bbox (matches the app: bbox is required).
import re

SPECIES_FACET_LIMIT = 1000

def _slug(text: str) -> str:
    text = re.sub(r'[^a-zA-Z0-9]+', '-', text.strip().lower())
    return text.strip('-') or 'place'

def _stable_id(entry: Dict, place: Optional[dict]) -> str:
    explicit = (entry.get('id') or '').strip().lower()
    if explicit:
        return _slug(explicit)
    if place and place.get('id'):
        return str(place['id'])
    return _slug(entry['query'])

def _place_checkpoint_path(entry: Dict, place: Optional[dict]) -> Path:
    return CHECKPOINT_DIR / f"{entry['kind']}__{_stable_id(entry, place)}.json"

def run_place(entry: Dict, *, species_facet_limit: int = SPECIES_FACET_LIMIT) -> Optional[dict]:
    query = entry['query']
    kind = entry['kind']
    place = resolve_place(query)
    checkpoint_path = _place_checkpoint_path(entry, place)
    if checkpoint_path.exists():
        return json.loads(checkpoint_path.read_text(encoding='utf-8'))

    if not place or not place.get('bbox'):
        record = {
            'query': query,
            'kind': kind,
            'id': entry.get('id'),
            'status': 'no-bbox',
            'place': place,
        }
        checkpoint_path.write_text(json.dumps(record, indent=2), encoding='utf-8')
        return record

    metrics = compute_place_metrics(bbox=place['bbox'], species_facet_limit=species_facet_limit)
    record = {
        'query': query,
        'kind': kind,
        'id': entry.get('id'),
        'note': entry.get('note'),
        'status': 'ok',
        'place': place,
        'metrics': metrics,
    }
    checkpoint_path.write_text(json.dumps(record, indent=2), encoding='utf-8')
    return record

def run_all(entries: List[Dict], *, species_facet_limit: int = SPECIES_FACET_LIMIT) -> List[dict]:
    results: List[dict] = []
    for i, entry in enumerate(entries, start=1):
        label = f"{entry['kind']}:{entry.get('id') or entry['query']}"
        try:
            record = run_place(entry, species_facet_limit=species_facet_limit)
        except Exception as exc:
            print(f'[{i}/{len(entries)}] {label}: FAILED -> {exc!r}')
            continue
        status = record.get('status') if record else 'missing'
        if i % 10 == 0 or status != 'ok' or i == len(entries):
            print(f'[{i}/{len(entries)}] {label}: {status}')
        if record:
            results.append(record)
    return results

results = run_all(PLACE_ENTRIES)
ok_results = [r for r in results if r.get('status') == 'ok']
print('OK places:', len(ok_results), '/ total queries:', len(PLACE_ENTRIES))

In [ ]:
# Assemble final output with per-cohort percentiles.
# Cohort = entry['kind'] ('city' vs 'country'). Cities are compared to cities,
# countries to countries — mixing them would be misleading.

from collections import defaultdict
from datetime import datetime, timezone

def assemble_output(ok_results: List[dict], baseline: Dict) -> Dict:
    rows: List[dict] = []
    for r in ok_results:
        m = r['metrics']
        place = r['place']
        sig = signature_species(
            local_top_species=m['topSpecies'],
            global_species_map=global_species_map,
            local_total=m['totalRecords'],
            global_total=global_total,
        )
        rows.append({
            'id': r.get('id'),
            'query': r['query'],
            'kind': r['kind'],
            'note': r.get('note'),
            'place': place,
            'metrics': m,
            'signatureSpecies': sig,
        })

    cohorts: Dict[str, List[dict]] = defaultdict(list)
    for row in rows:
        cohorts[row['kind']].append(row)

    for kind, cohort_rows in cohorts.items():
        rec_density = [
            r['metrics']['recordsPerKm2']
            for r in cohort_rows
            if r['metrics'].get('recordsPerKm2') is not None
        ]
        species_counts = [r['metrics']['uniqueSpecies'] for r in cohort_rows]
        threat_shares = [r['metrics']['threatenedShare'] for r in cohort_rows]
        for r in cohort_rows:
            r['percentiles'] = {
                'cohort': kind,
                'cohortSize': len(cohort_rows),
                'recordsPerKm2': (
                    percentile_rank(rec_density, r['metrics']['recordsPerKm2'])
                    if r['metrics'].get('recordsPerKm2') is not None else None
                ),
                'uniqueSpecies': percentile_rank(species_counts, r['metrics']['uniqueSpecies']),
                'threatenedShare': percentile_rank(threat_shares, r['metrics']['threatenedShare']),
            }

    cohort_meta = {kind: len(cohort_rows) for kind, cohort_rows in cohorts.items()}
    return {
        'meta': {
            'generatedAt': datetime.now(timezone.utc).isoformat(),
            'placeCount': len(rows),
            'cohorts': cohort_meta,
            'baselineTotal': baseline.get('total', 0),
            'baselineGeneratedAt': baseline.get('generatedAt'),
        },
        'rows': rows,
    }

output = assemble_output(ok_results, global_baseline)
OUTPUT_JSON_PATH.write_text(json.dumps(output, indent=2), encoding='utf-8')
print('wrote', OUTPUT_JSON_PATH)
print('cohorts:', output['meta']['cohorts'])
if output['rows']:
    sample = output['rows'][0]
    print('sample row keys:', list(sample.keys()))
    print('sample id/kind:', sample.get('id'), '/', sample.get('kind'))
    print('percentiles:', sample['percentiles'])

In [ ]:
# Quick sanity check of the exported JSON file.
import json
from collections import Counter

_PCT_KEYS = {'recordsPerKm2', 'uniqueSpecies', 'threatenedShare'}

if OUTPUT_JSON_PATH.exists():
    data = json.loads(OUTPUT_JSON_PATH.read_text(encoding='utf-8'))
    rows = data.get('rows', [])
    print('Loaded:', OUTPUT_JSON_PATH)
    print('Top-level keys:', sorted(data.keys()))
    print('Meta:', data.get('meta'))
    print('Row count:', len(rows))
    print('Cohort counts:', Counter(r.get('kind') for r in rows))

    if rows:
        sample = rows[0]
        print('Sample row keys:', sorted(sample.keys()))
        print('Sample query/kind:', sample.get('query'), '/', sample.get('kind'))
        print('Sample place label:', (sample.get('place') or {}).get('label'))
        print('Sample percentiles:', sample.get('percentiles'))
        bad = [
            k for k, v in (sample.get('percentiles') or {}).items()
            if k in _PCT_KEYS and isinstance(v, (int, float)) and not (0 <= v <= 1)
        ]
        print('Out-of-range percentile keys:', bad)
        sig = sample.get('signatureSpecies') or []
        print('Signature species count:', len(sig))
        if sig:
            print('Top signature ratio:', sig[0].get('overRepresentationRatio'))

## Scaling to a bigger run

- Append to `PLACE_QUERIES` and re-run the per-place cell — checkpoints make this resumable.
- Each place is cached as a single JSON file under `~/bee_around_precompute/checkpoints/`.
- Nominatim results are cached in `~/bee_around_precompute/nominatim_cache.json` (separate from GBIF results).
- The global baseline is cached in `~/bee_around_precompute/global_baseline.json`; delete it to force a refresh.
- To rerun a single place, delete its checkpoint file.